# Stat451: Machine Learning -- L02: kNN in Python

STAT 451: Intro to Machine Learning (Fall 2020)  

- Instructor: Sebastian Raschka (sraschka@wisc.edu)   
- Course website: http://pages.stat.wisc.edu/~sraschka/teaching/stat451-fs2020/

In [ ]:
# to install watermark (optional)
# uncomment the next line of code (and leave the "!")
#!pip install watermark

In [ ]:
# If you are using conda, you can also install watermark like this:
#!conda install watermark -y --channel conda-forge

In [ ]:
%load_ext watermark
%watermark  -d -u -a 'Sebastian Raschka' -v -p numpy,scipy,matplotlib,sklearn

## 0 - General Imports

I always find it helpful to put all the imports being used in the notebook into one cell somewhere at the top. This way, it is easy to tell what will be required for running the code in this notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1 - Load Dataset into a Pandas DataFrame

The next cell is actually not Python code but Bash terminal command that will show the first 10 lines of the [./iris.csv](./iris.csv) file. In Jupyter Notebook or Jupyer Lab you can run terminal commands by prepending "!" to a given line that is to be executed.

In [ ]:
!head iris.csv 

The next line of code will read the CSV file into a Pandas `DataFrame` object so that we have it in our current Python session and can work with it:

In [ ]:
df_iris = pd.read_csv('iris.csv')
df_iris.tail()

## 2 - Get Features into a NumPy Array

Recently, machine learning libraries such as scikit-learn started to support Pandas DataFrames. However, traditionally, it is customary to work with NumPy arrays. In fact, `DataFrame`s are build on top of NumPy arrays, so NumPy arrays are a bit lower-level. Btw. you can think of a NumPy array as a tensor, you can have scalars, vectors, matrices, etc. represented as NumPy arrays.

In the next code cell, we are accessing the NumPy array that is embedded in the DataFrame via the `.values` attribute. More specifically, we are assigning only the 2 feature columns `'PetalLength[cm]', 'PetalWidth[cm]'` to a variable `X`. Here, `X` references the NumPy array of shape 150x2 (you can think of it as a matrix).

In [ ]:
X = df_iris[['PetalLength[cm]', 'PetalWidth[cm]']].values
print(X.shape)

The next cell will show the first 5 entries (examples) in the dataset:

In [ ]:
X[:5, :]

## 3 - Get Class Labels into a NumPy Array

Similary to assigning the features to a new variable `X`, we are now assigning the class lables to a new variable, which we will call `y`. However, before we do that, let us map the class labels from a `string` representation into an `int` (integer) format. For this, we add a new column `ClassLabel` to the data frame:

In [ ]:
label_dict = {'Iris-setosa': 0,
              'Iris-versicolor': 1,
              'Iris-virginica': 2}

df_iris['ClassLabel'] = df_iris['Species'].map(label_dict)
df_iris.tail()

Then, similar to before, we reference the underlying NumPy array via the `.values` attribute and assign it to a variable `y`:

In [ ]:
y = df_iris['ClassLabel'].values
y[:10]

The next task is to divide the dataset into a training and a test set. Notice though that the first 10 entries are all zeros. This is because the dataset has not been shuffled yet. Hence, in Iris, the vector `y` will consists of 50 0's, 50 1's, and 50 2's (in that order).

It would thus be a bit problematic if we assigned the first 100 entries to a training set and the 50 last entries to the test set. Can you see why?



## 4 - Shuffle Dataset and Create Training and Test Subsets

In order to divide the dataset into a training and a test set, we are going to shuffle the dataset first. For illustration purposes, we will do this in multiple steps.

In the first step, we create an index array of our dataset: 

In [ ]:
indices = np.arange(y.shape[0])
indices

Next, we shuffle this index array. Note that we use a random seed such that other people can reproduce the results. This means that if someone else uses this code on a different computer, the person will get the exact same shuffle order.

In [ ]:
rnd = np.random.RandomState(123)
shuffled_indices = rnd.permutation(indices)
shuffled_indices

Now that we have the shuffled indices, we can use them to shuffle both the features `X` and `y` relative to each other -- we do this to preserve the correspondence between feature and label in `X` and `y`:

In [ ]:
X_shuffled, y_shuffled = X[shuffled_indices], y[shuffled_indices]

Finally, we can divide the dataset into training and test subsets:

In [ ]:
X_train, y_train = X_shuffled[:105], y_shuffled[:105]
X_test, y_test = X_shuffled[105:], y_shuffled[105:]

## 5 - Doing Steps 1-4 in Scikit-Learn

Note that we went through a lot of steps in order to load the dataset and to split it into training and test subsets. We did this so that you develop an intuition for the procedure. However, there are also convenient functions in scikit-learn that allow us to do the exact same things in much fewer lines of code. I.e., the following code will load the iris dataset and split it into 30% test data and 70% training data:

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split


iris = load_iris()
X, y = iris.data[:, 2:], iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.3,
                                                    random_state=123,
                                                    shuffle=True)

In [ ]:
print(f'X_train.shape: {X_train.shape}')
print(f'y_train.shape: {y_train.shape}')
print(f'X_test.shape: {X_test.shape}')
print(f'y_test.shape: {y_test.shape}')

## 6 - Plot Dataset

As you have probably already learned in other statistics classes, it is always a good idea to visualize your data before doing any further analyses. 

Below we are plotting all 4 features using a scatter plot matrix:

In [ ]:
from mlxtend.plotting import scatterplotmatrix


names = ['sepal length [cm]', 'sepal width [cm]',
         'petal length [cm]', 'petal width [cm]']

fig, axes = scatterplotmatrix(iris.data[y==0], figsize=(10, 8), alpha=0.5)
fig, axes = scatterplotmatrix(iris.data[y==1], fig_axes=(fig, axes), alpha=0.5)
fig, axes = scatterplotmatrix(iris.data[y==2], fig_axes=(fig, axes), alpha=0.5, names=names)

plt.tight_layout()
plt.show()

However, we are only working with 2 features, i.e., petal length and petal width (via `X, y = iris.data[:, 2:], iris.target` earlier). Below is a scatter plot matrix of these 2 features for `X_train`:

In [ ]:
names = ['petal length [cm]', 'petal width [cm]']

fig, axes = scatterplotmatrix(X_train[y_train==0], figsize=(10, 8), alpha=0.5, label="Setosa")
fig, axes = scatterplotmatrix(X_train[y_train==1], fig_axes=(fig, axes), alpha=0.5, label="Versicolor")
fig, axes = scatterplotmatrix(X_train[y_train==2], fig_axes=(fig, axes), alpha=0.5, names=names, label="Virginica")

fig.legend()

plt.tight_layout()
plt.show()

In this case, it may be overkill to use the scatterplot function; we can create a scatter plot using matplotlib directly:

In [ ]:
plt.scatter(X_train[y_train == 0, 0],
            X_train[y_train == 0, 1],
            marker='o',
            label='class 0 (Setosa)')

plt.scatter(X_train[y_train == 1, 0],
            X_train[y_train == 1, 1],
            marker='^',
            label='class 1 (Versicolor)')

plt.scatter(X_train[y_train == 2, 0],
            X_train[y_train == 2, 1],
            marker='s',
            label='class 2 (Virginica)')

plt.xlabel('petal length [cm]')
plt.ylabel('petal width [cm]')
plt.legend(loc='upper left')

plt.show()

Q: What can you learn about this dataset? Pay close attention to the two axes ...

## 7 - Fit k-Nearest Neighbor Model

Next, we are going to use the `KNeighborsClassifier` class from scikit-learn and instantiate a new KNN object, which we call `knn_model`. Then, we "fit" this model the training dataset:

In [ ]:
from sklearn.neighbors import KNeighborsClassifier


knn_model = KNeighborsClassifier(n_neighbors=3)
knn_model.fit(X_train, y_train)

Q: What distance measure is used by default?

## 8 - Use kNN Model to Make Predictions

After "fitting" the KNN model, we can now make predictions on new data points that the model has not seen before. The test set represents such data points. Of course, we have labels for the test set as well, but we only use these labels to evaluate the model -- i.e., we compare the predictions to the true labels in the test set in order to find out how good the model is:

In [ ]:
y_pred = knn_model.predict(X_test)

In [ ]:
num_correct_predictions = (y_pred == y_test).sum()
accuracy = (num_correct_predictions / y_test.shape[0]) * 100

# print('Test set accuracy: %.2f%%' % accuracy)

print(f'Test set accuracy: {accuracy:.2f}%')

In [ ]:
# print('Test set accuracy: %.2f%%' % (knn_model.score(X_test, y_test)*100))

print(f'Test set accuracy: {knn_model.score(X_test, y_test)*100:.2f}%')

## 9 - Visualize Decision Boundary

Usually, in machine learning, we work with datasets that have more then 2 feature variables. For educational purposes, however, we chose a very simple dataset consisting of only two features here (the petal length and the petal width of Iris flowers). If we have only two features, we can easily visualize the decision boundary of the model -- here, we will be using a convenience function I implemented in the MLxtend library:

In [ ]:
from mlxtend.plotting import plot_decision_regions


plot_decision_regions(X_train, y_train, knn_model)
plt.xlabel('petal length [cm]')
plt.ylabel('petal width [cm]')
plt.legend(loc='upper left')
plt.show()

In [ ]:
plot_decision_regions(X_test, y_test, knn_model)
plt.xlabel('petal length [cm]')
plt.ylabel('petal width [cm]')
plt.legend(loc='upper left')
plt.show()